In [ ]:
!pip install huggingface_hub boto3

In [ ]:
# ====== Create S3 client ======
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
import os

In [ ]:
# ====== CONFIGURE ======
S3_ENDPOINT_URL = "SET S3 API ENDPOINT" # Use the aws-compatible-storage-api Route
AWS_ACCESS_KEY_ID = "SET ACCESS KEY"
AWS_SECRET_ACCESS_KEY = "SET SECRET KEY"

In [ ]:
# Create the S3 client
client = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT_URL,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name="us-east-1",
    config=Config(s3={"addressing_style": "path"}),
)

In [ ]:
# ====== Download the model from Hugging Face and upload to S3 ======
import tempfile
import boto3
from huggingface_hub import hf_hub_download, list_repo_files

In [ ]:
# ====== CONFIGURE ======
repo_id = "meta-llama/Llama-3.1-8B-Instruct"  # Change to your model repo
revision = "main"
bucket_name = "your-bucket" # Your bucket name
huggingface_token = None # IF NEEDED ADD YOUR HUGGINGFACE TOKEN HERE

In [ ]:
# ====== DISABLE HF CACHE ======
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = tempfile.mkdtemp()

In [ ]:
files = list_repo_files(repo_id=repo_id, revision=revision, token=huggingface_token)

for file in files:
    print(f"Processing: {file}")
    try:
        # 1. Download to a temp file
        temp_path = hf_hub_download(
            repo_id=repo_id,
            filename=file,
            revision=revision,
            cache_dir=os.environ['HF_HOME'],
            local_dir_use_symlinks=False,
        )

        # 2. Upload to S3
        try:
            client.upload_file(temp_path, bucket_name, file)
            print(f"'{file}' is successfully uploaded as object to bucket '{bucket_name}'.")
        except ClientError as e:
            print("Error occurred: ", e)

        # 3. Delete file to free memory
        os.remove(temp_path)
        print(f"Deleted local file: {temp_path}")

    except Exception as e:
        print(f"Error processing {file}: {e}")